In [3]:
!pip install ultralytics opencv-python


In [ ]:
import cv2
from ultralytics import YOLO
from pathlib import Path
import shutil
import os
import requests

# Funkcja do pobrania modelu
def download_model(url, save_path):
    print(f"Pobieranie modelu z {url}...")
    response = requests.get(url, stream=True)
    with open(save_path, 'wb') as f:
        shutil.copyfileobj(response.raw, f)
    print(f"Model zapisany jako {save_path}")

# Ścieżki bazowe (względne)
NOTEBOOK_DIR = Path.cwd()  # zakładamy, że to notebook
DATA_DIR = NOTEBOOK_DIR.parent / "data" / "processed"

# URL modelu i lokalna ścieżka
model_url = "https://github.com/akanametov/yolov8-face/releases/download/v0.0.0/yolov8n-face.pt"
model_path = NOTEBOOK_DIR / "yolov8n-face.pt"

# Pobierz model jeśli nie istnieje
if not os.path.exists(model_path):
    download_model(model_url, model_path)

# Załaduj model
model = YOLO(str(model_path))  # Konwersja na string dla kompatybilności

# Ścieżki do folderów z danymi
folders = [
    DATA_DIR / "test/none",
    DATA_DIR / "train/none",
    DATA_DIR / "val/none"
]

for input_folder in folders:
    # Utwórz folder wyjściowy jeśli nie istnieje
    output_folder = input_folder / "faces_detected"
    os.makedirs(output_folder, exist_ok=True)

    print(f"\nPrzetwarzanie folderu: {input_folder}")
    print(f"Znaleziono {len(list(input_folder.glob('*.jpeg')))} zdjęć do przetworzenia")

    # Przetwarzaj tylko pliki JPEG
    for img_path in input_folder.glob("*.jpeg"):
        # Pomijaj ukryte pliki
        if img_path.name.startswith('.'):
            continue

        try:
            # Wczytaj obraz
            img = cv2.imread(str(img_path))
            if img is None:
                print(f"Nie można wczytać obrazu: {img_path.name}")
                continue

            # Wykrywanie twarzy
            results = model(img)

            # Sprawdź czy znaleziono twarze
            face_found = any(len(r.boxes) > 0 for r in results)

            if face_found:
                shutil.move(str(img_path), str(output_folder / img_path.name))

        except Exception as e:
            print(f"! Błąd podczas przetwarzania {img_path.name}: {str(e)}")

print("\nPrzetwarzanie zakończone!")


Przetwarzanie folderu: C:\Users\user\PycharmProjects\classification\Gender_Detection\data\processed\test\none
Znaleziono 1623 zdjęć do przetworzenia

0: 448x640 (no detections), 124.3ms
Speed: 8.5ms preprocess, 124.3ms inference, 6.1ms postprocess per image at shape (1, 3, 448, 640)
× Brak twarzy w: adler-bird-bird-of-prey-raptor-53587.jpeg

0: 416x640 28 faces, 60.6ms
Speed: 3.7ms preprocess, 60.6ms inference, 6.8ms postprocess per image at shape (1, 3, 416, 640)
✓ Twarz wykryta w: art-school-of-athens-raphael-italian-painter-fresco-159862.jpeg

0: 544x640 8 faces, 76.4ms
Speed: 4.1ms preprocess, 76.4ms inference, 1.3ms postprocess per image at shape (1, 3, 544, 640)
✓ Twarz wykryta w: automotive-defect-broken-car-wreck-78793.jpeg

0: 480x640 (no detections), 65.8ms
Speed: 3.2ms preprocess, 65.8ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)
× Brak twarzy w: bald-eagle-cub-head-portrait-42268.jpeg

0: 448x640 1 face, 49.9ms
Speed: 3.6ms preprocess, 49.9ms inferenc

In [4]:
!pip install insightface opencv-python onnxruntime


   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.7 MB 5.6 MB/s eta 0:00:03
   --------- ------------------------------ 3.1/12.7 MB 9.2 MB/s eta 0:00:02
   --------------- ------------------------ 5.0/12.7 MB 9.2 MB/s eta 0:00:01
   ----------------------- ---------------- 7.3/12.7 MB 9.9 MB/s eta 0:00:01
   ----------------------------- ---------- 9.4/12.7 MB 10.0 MB/s eta 0:00:01
   ------------------------------------- -- 11.8/12.7 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 12.7/12.7 MB 10.1 MB/s eta 0:00:00

   -------- ------------------------------- 1/5 [pyreadline3]
   ---------------- ----------------------- 2/5 [humanfriendly]
   -------------------------------- ------- 4/5 [onnxruntime]
   -------------------------------- ------- 4/5 [onnxruntime]
   -------------------------------- ------- 4/5 [onnxruntime]
   -------------------------------- ------- 4/5 [onnxruntime]
   --------

In [6]:
import cv2
import insightface
from insightface.app import FaceAnalysis
from pathlib import Path
import shutil
import os

# Ścieżki bazowe (względne)
NOTEBOOK_DIR = Path.cwd()  # zakładamy, że to notebook
DATA_DIR = NOTEBOOK_DIR.parent / "data" / "processed"

app = FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0) #if 0 then cpu if 1 then gpu


# Ścieżki do folderów z danymi
folders = [
    DATA_DIR / "test/none",
    DATA_DIR / "train/none",
    DATA_DIR / "val/none"
]

for input_folder in folders:
    # Utwórz folder wyjściowy jeśli nie istnieje
    output_folder = input_folder / "faces_detected"
    output_folder.mkdir(parents=True, exist_ok=True)

    print(f"\nPrzetwarzanie folderu: {input_folder}")
    jpeg_files = list(input_folder.glob('*.jpeg'))
    print(f"Znaleziono {len(jpeg_files)} zdjęć do przetworzenia")

    # Przetwarzaj tylko pliki JPEG
    for img_path in jpeg_files:
        if img_path.name.startswith('.'):
            continue

        try:
            img = cv2.imread(str(img_path))
            if img is None:
                print(f"Nie można wczytać obrazu: {img_path.name}")
                continue

            faces = app.get(img)

            if faces:
                shutil.move(str(img_path), str(output_folder / img_path.name))

        except Exception as e:
            print(f"! Błąd podczas przetwarzania {img_path.name}: {str(e)}")

print("\nPrzetwarzanie zakończone!")


download_path: C:\Users\user/.insightface\models\buffalo_l


100%|██████████| 281857/281857 [00:36<00:00, 7761.93KB/s] 
C:\Users\user\PycharmProjects\classification\Gender_Detection\venv\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:121: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\user/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\user/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\user/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\user/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\user/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127